# Product Quantization: From Scratch, With Pictures

We'll build PQ step by step on a toy 8-dimensional dataset, visualize every stage, and see how close the approximate distances get to the real ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)

# Color palette we'll reuse everywhere
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
BG_COLOR = '#1a1a2e'
GRID_COLOR = '#2a2a4a'
TEXT_COLOR = '#e0e0e0'

plt.rcParams.update({
    'figure.facecolor': BG_COLOR,
    'axes.facecolor': BG_COLOR,
    'axes.edgecolor': GRID_COLOR,
    'axes.labelcolor': TEXT_COLOR,
    'text.color': TEXT_COLOR,
    'xtick.color': TEXT_COLOR,
    'ytick.color': TEXT_COLOR,
    'grid.color': GRID_COLOR,
    'font.size': 11,
    'figure.dpi': 120
})

## 1. Generate the Dataset

In [ ]:
N = 1000   # number of vectors
d = 8      # dimensions
m = 4      # sub-vectors
k = 4      # centroids per codebook
ds = d // m  # dims per sub-vector

data = np.random.rand(N, d).astype(np.float32)

print(f"Dataset shape: {data.shape}")
print(f"Raw size: {data.nbytes:,} bytes")
print(f"\nPQ config: {m} sub-vectors × {ds} dims each, {k} centroids per codebook")
print(f"Compressed size per vector: {m} bytes (one code per sub-vector)")
print(f"Compression ratio: {(d * 4) / m}x")
print(f"\nFirst 5 vectors:")
data[:5]

## 2. Split Vectors into Sub-vectors

Each 8-dim vector gets chopped into 4 pieces of 2 dims each. Let's visualize what one vector looks like after splitting.

In [ ]:
# Split into sub-vectors
sub_vectors = [data[:, i*ds:(i+1)*ds] for i in range(m)]

x = data[0]
print(f"Original vector: {x}")
print()
for i in range(m):
    print(f"Sub-vector {i} (dims {i*ds}, {i*ds+1}): {x[i*ds:(i+1)*ds]}")

In [ ]:
# Visualize: the original vector as a colored bar chart
fig, ax = plt.subplots(figsize=(10, 3))

bar_colors = [COLORS[i // ds] for i in range(d)]
bars = ax.bar(range(d), x, color=bar_colors, edgecolor='white', linewidth=0.5, width=0.8)

ax.set_xlabel('Dimension')
ax.set_ylabel('Value')
ax.set_title('One Vector Split into 4 Sub-vectors (color = sub-vector group)', fontsize=13, fontweight='bold')
ax.set_xticks(range(d))
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

# Add sub-vector brackets
for i in range(m):
    ax.axvspan(i*ds - 0.45, (i+1)*ds - 0.55, alpha=0.08, color=COLORS[i])
    ax.text(i*ds + ds/2 - 0.5, 1.05, f'Sub-vector {i}', ha='center', fontsize=9, color=COLORS[i], fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Train Codebooks with K-Means

For each sub-vector position, we cluster all 1000 data points in that 2D slice. This gives us 4 centroids per position = our "codebook."

In [ ]:
codebooks = []
assignments = []  # which centroid each point got assigned to

for i in range(m):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(sub_vectors[i])
    codebooks.append(km.cluster_centers_)
    assignments.append(km.labels_)
    
    print(f"Codebook {i} (dims {i*ds}, {i*ds+1}):")
    for j, c in enumerate(km.cluster_centers_):
        print(f"  C{j}: [{c[0]:.3f}, {c[1]:.3f}]")
    print()

In [ ]:
# Visualize: 2D scatter plot for each sub-vector position showing clusters and centroids
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

centroid_colors = ['#ff6b6b', '#4ecdc4', '#ffe66d', '#a29bfe']

for i, ax in enumerate(axes):
    sv = sub_vectors[i]
    labels = assignments[i]
    centroids = codebooks[i]
    
    # Plot each cluster
    for j in range(k):
        mask = labels == j
        ax.scatter(sv[mask, 0], sv[mask, 1], 
                   c=centroid_colors[j], alpha=0.25, s=15, edgecolors='none')
    
    # Plot centroids as big stars
    for j, c in enumerate(centroids):
        ax.scatter(c[0], c[1], c=centroid_colors[j], s=250, marker='*', 
                   edgecolors='white', linewidth=1.2, zorder=5)
        ax.annotate(f'C{j}', (c[0], c[1]), textcoords="offset points", 
                    xytext=(8, 8), fontsize=9, fontweight='bold', color=centroid_colors[j])
    
    # Highlight our target vector's sub-vector
    target = x[i*ds:(i+1)*ds]
    ax.scatter(target[0], target[1], c='white', s=100, marker='D', 
               edgecolors=COLORS[i], linewidth=2, zorder=6)
    ax.annotate('x', (target[0], target[1]), textcoords="offset points",
                xytext=(8, -8), fontsize=10, fontweight='bold', color='white')
    
    ax.set_title(f'Codebook {i}\n(dims {i*ds}, {i*ds+1})', color=COLORS[i], fontweight='bold')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.2)
    ax.set_aspect('equal')

fig.suptitle('K-Means Clusters per Sub-vector Position (★ = centroid, ◆ = our vector)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Encode the Vector

For each sub-vector, find the nearest centroid. Replace the sub-vector with that centroid's index.

In [ ]:
codes = []
print(f"Original vector: {x}\n")

for i in range(m):
    sub = x[i*ds:(i+1)*ds]
    dists = np.linalg.norm(codebooks[i] - sub, axis=1)
    code = np.argmin(dists)
    codes.append(code)
    print(f"Sub-vector {i}: {sub} → C{code} = {codebooks[i][code]} (dist = {dists[code]:.4f})")

codes = np.array(codes, dtype=np.uint8)
print(f"\nEncoded vector: {codes}")
print(f"Original size:   {x.nbytes} bytes")
print(f"Compressed size: {codes.nbytes} bytes")
print(f"Compression:     {x.nbytes // codes.nbytes}x")

In [ ]:
# Visualize: original vector vs reconstructed (from centroids) vector
reconstructed = np.concatenate([codebooks[i][codes[i]] for i in range(m)])

fig, ax = plt.subplots(figsize=(10, 4))

width = 0.35
positions = np.arange(d)

bars1 = ax.bar(positions - width/2, x, width, label='Original', 
               color=[COLORS[i // ds] for i in range(d)], alpha=0.9, edgecolor='white', linewidth=0.5)
bars2 = ax.bar(positions + width/2, reconstructed, width, label='Reconstructed (from centroids)', 
               color=[COLORS[i // ds] for i in range(d)], alpha=0.4, edgecolor='white', linewidth=0.5,
               hatch='//')

ax.set_xlabel('Dimension')
ax.set_ylabel('Value')
ax.set_title('Original vs Reconstructed Vector (the quantization error)', fontsize=13, fontweight='bold')
ax.set_xticks(positions)
ax.set_ylim(0, 1.2)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Show the total reconstruction error
recon_error = np.sum((x - reconstructed) ** 2)
ax.text(0.02, 0.95, f'Reconstruction error (L2²): {recon_error:.4f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#2a2a4a', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Visualize: show which centroid was picked in each 2D sub-space
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for i, ax in enumerate(axes):
    centroids = codebooks[i]
    target = x[i*ds:(i+1)*ds]
    chosen = codes[i]
    
    # Plot all centroids
    for j, c in enumerate(centroids):
        alpha = 1.0 if j == chosen else 0.3
        size = 300 if j == chosen else 150
        ax.scatter(c[0], c[1], c=centroid_colors[j], s=size, marker='*', 
                   edgecolors='white', linewidth=1.2, alpha=alpha, zorder=5)
        ax.annotate(f'C{j}', (c[0], c[1]), textcoords="offset points",
                    xytext=(10, 8), fontsize=10, fontweight='bold', 
                    color=centroid_colors[j], alpha=alpha)
    
    # Plot the target sub-vector
    ax.scatter(target[0], target[1], c='white', s=120, marker='D', 
               edgecolors=COLORS[i], linewidth=2, zorder=6)
    
    # Draw arrow from target to chosen centroid
    chosen_c = centroids[chosen]
    ax.annotate('', xy=(chosen_c[0], chosen_c[1]), xytext=(target[0], target[1]),
                arrowprops=dict(arrowstyle='->', color='white', lw=2, 
                                connectionstyle='arc3,rad=0.1'))
    
    dist = np.linalg.norm(target - chosen_c)
    mid = (target + chosen_c) / 2
    ax.text(mid[0], mid[1] + 0.08, f'd={dist:.3f}', ha='center', fontsize=9, 
            color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=COLORS[i], alpha=0.7))
    
    ax.set_title(f'Sub-vector {i} → C{chosen}', color=COLORS[i], fontweight='bold', fontsize=12)
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(alpha=0.2)
    ax.set_aspect('equal')

fig.suptitle('Encoding: Each Sub-vector Snaps to Its Nearest Centroid', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Query Time: Build the Lookup Tables

In [ ]:
query = np.random.rand(d).astype(np.float32)
print(f"Query vector: {query}\n")

# Build lookup tables
tables = np.zeros((m, k))
for i in range(m):
    q_sub = query[i*ds:(i+1)*ds]
    for j in range(k):
        diff = q_sub - codebooks[i][j]
        tables[i][j] = np.sum(diff ** 2)
    print(f"Table {i} (query sub = {q_sub}):")
    for j in range(k):
        marker = ' ◄ LOOKUP' if j == codes[i] else ''
        print(f"  dist to C{j}: {tables[i][j]:.4f}{marker}")
    print()

In [ ]:
# Visualize: the lookup tables as a heatmap
fig, ax = plt.subplots(figsize=(8, 4))

im = ax.imshow(tables, cmap='YlOrRd', aspect='auto')

# Annotate each cell
for i in range(m):
    for j in range(k):
        color = 'white' if tables[i][j] > 0.3 else 'black'
        weight = 'bold' if j == codes[i] else 'normal'
        border = '  ' if j != codes[i] else ''
        ax.text(j, i, f'{tables[i][j]:.3f}', ha='center', va='center', 
                fontsize=11, color=color, fontweight=weight)
        # Highlight the looked-up cell
        if j == codes[i]:
            rect = plt.Rectangle((j-0.5, i-0.5), 1, 1, linewidth=3, 
                                  edgecolor='white', facecolor='none')
            ax.add_patch(rect)

ax.set_xticks(range(k))
ax.set_xticklabels([f'C{j}' for j in range(k)])
ax.set_yticks(range(m))
ax.set_yticklabels([f'Table {i}' for i in range(m)])
ax.set_xlabel('Centroid Index')
ax.set_title('Distance Lookup Tables (white boxes = values used for our vector)', 
             fontsize=12, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Squared L2 Distance', color=TEXT_COLOR)
cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=TEXT_COLOR)

plt.tight_layout()
plt.show()

## 6. Approximate vs Exact Distance

In [ ]:
approx_dist = sum(tables[i][codes[i]] for i in range(m))
exact_dist = np.sum((query - x) ** 2)

print(f"PQ lookup:  ", end="")
parts = [f"Table{i}[{codes[i]}]={tables[i][codes[i]]:.4f}" for i in range(m)]
print(" + ".join(parts))
print(f"          = {approx_dist:.4f}")
print(f"\nExact dist: {exact_dist:.4f}")
print(f"Error:      {abs(approx_dist - exact_dist):.4f}")
print(f"Relative:   {abs(approx_dist - exact_dist) / exact_dist * 100:.1f}%")

## 7. Scale Test: PQ Accuracy Across the Whole Dataset

Let's encode ALL 1000 vectors and compare approximate vs exact distances for all of them against our query.

In [ ]:
# Encode all vectors
all_codes = np.zeros((N, m), dtype=np.uint8)
for i in range(m):
    sv = sub_vectors[i]
    for n in range(N):
        dists = np.linalg.norm(codebooks[i] - sv[n], axis=1)
        all_codes[n, i] = np.argmin(dists)

# Compute approximate distances using lookup tables
approx_dists = np.zeros(N)
for n in range(N):
    approx_dists[n] = sum(tables[i][all_codes[n, i]] for i in range(m))

# Compute exact distances
exact_dists = np.sum((data - query) ** 2, axis=1)

print(f"Encoded {N} vectors into {all_codes.nbytes:,} bytes (was {data.nbytes:,} bytes)")
print(f"Mean absolute error: {np.mean(np.abs(approx_dists - exact_dists)):.4f}")
print(f"Mean relative error: {np.mean(np.abs(approx_dists - exact_dists) / exact_dists) * 100:.1f}%")

In [ ]:
# Visualize: scatter plot of approximate vs exact distances
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: Approximate vs Exact scatter
ax = axes[0]
ax.scatter(exact_dists, approx_dists, alpha=0.3, s=10, c='#4ecdc4', edgecolors='none')
lims = [0, max(exact_dists.max(), approx_dists.max()) * 1.05]
ax.plot(lims, lims, '--', color='#ff6b6b', linewidth=2, label='Perfect agreement')
ax.set_xlabel('Exact Squared L2 Distance')
ax.set_ylabel('Approximate (PQ) Distance')
ax.set_title('PQ Approximation Quality', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')

# Plot 2: Do the top-k results agree?
ax = axes[1]
k_vals = [1, 5, 10, 20, 50, 100]
recall_at_k = []
for kk in k_vals:
    true_topk = set(np.argsort(exact_dists)[:kk])
    pq_topk = set(np.argsort(approx_dists)[:kk])
    recall = len(true_topk & pq_topk) / kk
    recall_at_k.append(recall)
    print(f"Recall@{kk:3d}: {recall:.2%}")

ax.bar(range(len(k_vals)), recall_at_k, color='#a29bfe', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(k_vals)))
ax.set_xticklabels([f'@{kk}' for kk in k_vals])
ax.set_ylabel('Recall')
ax.set_ylim(0, 1.1)
ax.set_title('Recall@K: Does PQ Find the Right Neighbors?', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for i, v in enumerate(recall_at_k):
    ax.text(i, v + 0.03, f'{v:.0%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Bonus: What Happens When We Change k (Number of Centroids)?

More centroids = better approximation but larger codes.

In [ ]:
k_options = [2, 4, 8, 16, 32, 64]
mean_errors = []
bits_per_vector = []

for k_test in k_options:
    # Train codebooks
    cb = []
    for i in range(m):
        km = KMeans(n_clusters=k_test, random_state=42, n_init=5)
        km.fit(sub_vectors[i])
        cb.append(km.cluster_centers_)
    
    # Encode
    tc = np.zeros((N, m), dtype=np.int32)
    for i in range(m):
        for n in range(N):
            dists = np.linalg.norm(cb[i] - sub_vectors[i][n], axis=1)
            tc[n, i] = np.argmin(dists)
    
    # Build tables and score
    t = np.zeros((m, k_test))
    for i in range(m):
        q_sub = query[i*ds:(i+1)*ds]
        for j in range(k_test):
            t[i][j] = np.sum((q_sub - cb[i][j]) ** 2)
    
    ad = np.array([sum(t[i][tc[n, i]] for i in range(m)) for n in range(N)])
    
    err = np.mean(np.abs(ad - exact_dists) / exact_dists) * 100
    bits = m * np.ceil(np.log2(k_test))
    mean_errors.append(err)
    bits_per_vector.append(bits)
    print(f"k={k_test:3d} | {bits:5.0f} bits/vector | mean relative error: {err:.1f}%")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = '#e74c3c'
color2 = '#3498db'

ax1.plot(k_options, mean_errors, 'o-', color=color1, linewidth=2.5, markersize=8, label='Mean Relative Error %')
ax1.set_xlabel('Number of Centroids (k)', fontsize=12)
ax1.set_ylabel('Mean Relative Error (%)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xscale('log', base=2)
ax1.set_xticks(k_options)
ax1.set_xticklabels([str(k) for k in k_options])
ax1.grid(alpha=0.2)

ax2 = ax1.twinx()
ax2.bar([x * 1.08 for x in range(len(k_options))], bits_per_vector, 
        color=color2, alpha=0.3, width=0.6)
# Fix: use actual x positions matching the log scale
ax2.set_ylabel('Bits per Vector', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('The Tradeoff: More Centroids = Better Accuracy but Bigger Codes', 
              fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nThe sweet spot in production: k=256 (8 bits per sub-vector, 1 byte)")
print("Good enough accuracy, dead simple storage (one byte per code).")

## Summary

What we just did:
1. **Split** each 8-dim vector into 4 sub-vectors of 2 dims
2. **Clustered** each sub-vector position with k-means (4 centroids each)
3. **Encoded** each vector as 4 centroid IDs (4 bytes instead of 32)
4. **Queried** using precomputed lookup tables (4 lookups + 3 additions)
5. **Verified** that the approximate distances closely match exact distances

Scale this to d=3072, m=384, k=256 and you go from 12 KB per vector to 384 bytes. A 32x compression. A terabyte database fits in 37 GB of RAM.